## Carregando Dados e Bibliotecas necessárias.

In [ ]:
import pandas as pd
import numpy as np

data_layer_filepath = '../../data_layer/'

df = pd.read_csv(data_layer_filepath + 'raw/airbnb-dataset.csv', low_memory=False)
print("Dataset carregado com sucesso!")
df.head()

Dataset carregado com sucesso!


,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,country,...,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,house_rules,license
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,...,$193,10.0,9.0,10/19/2021,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,NaN
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,...,$28,30.0,45.0,5/21/2022,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,NaN
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,NaN,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,...,$124,3.0,0.0,NaN,NaN,5.0,1.0,352.0,"I encourage you to use my kitchen, cooking and...",NaN
3,1002755,NaN,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,...,$74,30.0,270.0,7/5/2019,4.64,4.0,1.0,322.0,NaN,NaN
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,United States,...,$41,10.0,9.0,11/19/2018,0.10,3.0,1.0,289.0,"Please no smoking in the house, porch or on th...",NaN


# Tratamento dos Dados.

## Padronização dos Nomes das Colunas.

In [ ]:
df.rename(
    columns={col: col.lower().replace(' ', '_') for col in df.columns},
    inplace=True
)
print(df.columns)

Index(['id', 'name', 'host_id', 'host_identity_verified', 'host_name',
       'neighbourhood_group', 'neighbourhood', 'lat', 'long', 'country',
       'country_code', 'instant_bookable', 'cancellation_policy', 'room_type',
       'construction_year', 'price', 'service_fee', 'minimum_nights',
       'number_of_reviews', 'last_review', 'reviews_per_month',
       'review_rate_number', 'calculated_host_listings_count',
       'availability_365', 'house_rules', 'license'],
      dtype='object')


## Remoção de Colunas Desnecessárias

Como quase todas as tuplas de **license** estavam como nulas, optamos por não trabalhar com essa coluna. Além disso, optamos por remover as colunas de **Country** e **Country_code** visto que sabemos que todas se enquadram no Estados Unidos e possuem o códido do país como "US".

In [ ]:
cols_to_drop = ['country', 'country_code', 'license']
df.drop(columns=cols_to_drop, inplace=True)

for col in cols_to_drop:
    if col not in df.columns:
        print(f"Coluna {col} deletada!")

print("Colunas: ")
print(df.columns)

Coluna country deletada!
Coluna country_code deletada!
Coluna license deletada!
Colunas: 
Index(['id', 'name', 'host_id', 'host_identity_verified', 'host_name',
       'neighbourhood_group', 'neighbourhood', 'lat', 'long',
       'instant_bookable', 'cancellation_policy', 'room_type',
       'construction_year', 'price', 'service_fee', 'minimum_nights',
       'number_of_reviews', 'last_review', 'reviews_per_month',
       'review_rate_number', 'calculated_host_listings_count',
       'availability_365', 'house_rules'],
      dtype='object')


## Correções dos Tipos de Dados.

1) **price** e **service_fee** possuem o caracter especial "$" e estão como object. Com isso, iremos altera-las para o tipo númerico (Float)

In [ ]:
money_columns = ['price', 'service_fee']

print("Tipos de dados antes da correção:")
print(df[money_columns].dtypes)

for col in money_columns:
    df[col] = df[col].str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip().astype(float)

print("Tipos de dados corrigidos:")
print(df[money_columns].dtypes)

Tipos de dados antes da correção:
price          object
service_fee    object
dtype: object
Tipos de dados corrigidos:
price          float64
service_fee    float64
dtype: object


2) **construction_year**, **availability_365** e **calculated_host_listings_count**: Não faria sentido estar sendo guardado em Float já que era o ano de construção. Ou seja, nunca viria um número decimal. O mesmo vale para availability_365 e para calculated_host_listings_count.

In [ ]:
floats_to_ints = ['availability_365', 'construction_year','calculated_host_listings_count']

for col in floats_to_ints:
    print(f"Tipo de dado da chave {col} antes da correção: {df[col].dtype}")

    df.dropna(subset=[col], inplace=True)
    df[col] = df[col].astype('int64')

    print(f"Tipo de dado da chave {col} após a correção: {df[col].dtype}")

Tipo de dado da chave availability_365 antes da correção: float64
Tipo de dado da chave availability_365 após a correção: int64
Tipo de dado da chave construction_year antes da correção: float64
Tipo de dado da chave construction_year após a correção: int64
Tipo de dado da chave calculated_host_listings_count antes da correção: float64
Tipo de dado da chave calculated_host_listings_count após a correção: int64


3) **host_identity_verified**: A coluna host_identity_verified pode ser transformada em booleano (True/False), o que é mais eficiente e semanticamente correto.

In [ ]:
df['host_identity_verified'] = df['host_identity_verified'].astype(str).str.lower().str.strip()

to_replace = {
    'verified': True,
    'unconfirmed': False
}

df['host_identity_verified'] = df['host_identity_verified'].replace(to_replace).astype(bool)


print("Tipo de dado da coluna APÓS o tratamento:")
print(df['host_identity_verified'].dtype)
print("\nValores únicos na coluna APÓS o tratamento:")
print(df['host_identity_verified'].unique())
print("\nContagem de valores na coluna APÓS o tratamento:")
print(df['host_identity_verified'].value_counts(dropna=False))

Tipo de dado da coluna APÓS o tratamento:
bool

Valores únicos na coluna APÓS o tratamento:
[False  True]

Contagem de valores na coluna APÓS o tratamento:
host_identity_verified
True     50916
False    50724
Name: count, dtype: int64


4. **Colunas object**: algumas colunas tem o tipo misto `object`. Na análise realizada na camada bronze, concluímos que essas colunas devem ter o tipo `string`. Além disso, a coluna `instant_bookable` deveria ter o tipo `bool`. Por fim, a coluna `last_review` deveria ter um tipo de dados que melhor representa uma data.

In [ ]:
string_cols = [
    'name',
    'host_name',
    'neighbourhood_group',
    'neighbourhood',
    'cancellation_policy',
    'room_type',
    'house_rules',
]

for col in string_cols:
    df[col] = df[col].astype('string')

df['instant_bookable'] = df['instant_bookable'].astype(bool)

df['last_review'] = pd.to_datetime(df['last_review'])

5. **minimum_nights**: esta coluna é do tipo `float64`, mas deveria ter um tipo inteiro, visto que trata da quantidade de noites mínimas que um interessado deve passar num lugar anunciado.

In [ ]:
df.dropna(subset=['minimum_nights'], inplace=True)
df['minimum_nights'] = df['minimum_nights'].astype('int64')

## Correção de Inconsistência nos Dados.

### Correção nos erros de digitação no nome dos bairros que apresentavam "brookln" e "manhatan"

In [ ]:

df['neighbourhood_group'] = df['neighbourhood_group'].replace({
    'brookln': 'Brooklyn',
    'manhatan': 'Manhattan'
})


print(df['neighbourhood_group'].unique())

<StringArray>
['Brooklyn', 'Manhattan', <NA>, 'Queens', 'Staten Island', 'Bronx']
Length: 6, dtype: string


### Tratamento de Valores Ausentes.


1) Remoção de Anúncios sem preço.

In [ ]:

df.dropna(subset=['price', 'service_fee'], inplace=True)

print(f"Valores nulos em 'price' após remoção: {df['price'].isnull().sum()}")

Valores nulos em 'price' após remoção: 0


2) Criação de Coluna booleana para house_rules: Como metade dos valores é nulo, iremos criar uma nova coluna para indicar se essa "casa" possui ou não regras definidas.

In [ ]:

df['has_house_rules'] = df['house_rules'].notna()

# Podemos agora remover a coluna original se o conteúdo de texto não for usado
# df_silver.drop(columns=['house_rules'], inplace=True)


print(df['has_house_rules'].value_counts())

has_house_rules
False    51240
True     49544
Name: count, dtype: int64


3) Preenchimento dos poucos anúncios sem nome (ou sem nome de host) com "Sem nome informado"

In [ ]:
for col in ['name', 'host_name']:
    df[col] = df[col].fillna('Sem nome informado')

4) Remoção dos demais **nans**

In [ ]:
nans_to_drop = [
    'neighbourhood', 
    'neighbourhood_group', 
    'lat', 
    'long',
    'host_identity_verified',
    'minimum_nights',
]

df.dropna(subset=nans_to_drop, inplace=True)

## Tratamentos de Valores invalidados.

Foi identificado alguns valores negativos na coluna **availability_365** que não fazem sentido com o escopo.

In [ ]:
df = df[df['availability_365'] >= 0]
print(df['availability_365'].min())

0


# Tratamentos de Dados Duplicados


In [ ]:
print("--- DIAGNÓSTICO E REMOÇÃO DE DUPLICADOS ---")
print(f"Número de linhas TOTAIS no DataFrame inicial: {len(df)}")
print("-" * 45)


linhas_duplicadas = df[df.duplicated(subset=['id'], keep=False)]

if not linhas_duplicadas.empty:
    num_ids_unicos_duplicados = linhas_duplicadas['id'].nunique()
    print(f"Encontrados {num_ids_unicos_duplicados} IDs únicos que se repetem.")
    print(f"Esses IDs correspondem a um total de {len(linhas_duplicadas)} linhas no DataFrame.")
else:
    print("Nenhuma duplicata encontrada.")

df_sem_duplicados = df.drop_duplicates(subset=['id'], keep='first')

print(f"\nNúmero de linhas APÓS a remoção: {len(df_sem_duplicados)}")
print(f"Cálculo da operação: {len(df)} (linhas iniciais) - {df.duplicated(subset=['id']).sum()} (ocorrências extras) = {len(df_sem_duplicados)}")
print("-" * 45)

duplicados_apos_limpeza = df_sem_duplicados.duplicated(subset=['id']).sum()

print("VERIFICAÇÃO FINAL:")
print(f"Número de IDs duplicados no DataFrame final: {duplicados_apos_limpeza}")

if duplicados_apos_limpeza == 0:
    print("\n✅ SUCESSO! A remoção de duplicados foi confirmada com sucesso.")
else:
    print("\n⚠️ ATENÇÃO! Ainda existem duplicados no DataFrame.")


## Dicionario - Silver

In [ ]:
print("--- Amostra Aleatória de 20 Linhas do DataFrame 'df_silver' ---")

# O .sample(20) pega 20 linhas aleatórias do DataFrame.
# O .reset_index(drop=True) é para a visualização ficar mais limpa, sem o índice antigo.
display(df.sample(20).reset_index(drop=True))


print("\n\n--- Resumo das Informações (Info) ---")
df.info()

--- Amostra Aleatória de 20 Linhas do DataFrame 'df_silver' ---


,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,instant_bookable,...,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365,house_rules,has_house_rules
0,19101312,"Charming Neighborhood, Charming Studio to Share",8881171415,True,Lillian,Manhattan,Upper West Side,40.78099,-73.97933,True,...,111.0,1,13.0,2019-07-04,1.06,5.0,1,331,No smoking or partying in the apartment.,True
1,1538172,THERE'S NO PLACE LIKE HOME.........,3468967958,True,Cunningham,Manhattan,Harlem,40.80151,-73.95220,False,...,135.0,2,70.0,2019-06-03,0.80,5.0,4,275,<NA>,False
2,56804623,Taaffe Loft,93871023441,True,Michael,Brooklyn,Bedford-Stuyvesant,40.69121,-73.95877,True,...,112.0,1,0.0,NaT,NaN,1.0,1,0,just act like my friend in building and don't ...,True
3,2531759,"AMAZING, HUGE NEW YORK LOFT!",79474011647,False,Habib,Brooklyn,Clinton Hill,40.68639,-73.96240,False,...,239.0,3,38.0,2019-06-25,0.54,4.0,1,317,<NA>,False
4,10378285,"Peaceful, Light-filled Room - Greenpoint, BK",36520372602,False,Michele,Brooklyn,Greenpoint,40.73265,-73.95393,False,...,84.0,7,38.0,2019-01-23,1.08,5.0,1,223,<NA>,False
5,20918379,Gorgeous Bedroom in the heart of Bushwick Him-...,65247718044,False,Nina,Brooklyn,Bushwick,40.69952,-73.91868,True,...,220.0,30,1.0,2019-05-04,0.45,2.0,17,342,Follow the Golden Rule of treating others (hom...,True
6,9170958,Single room in Bushwick w/backyard,31635172895,True,Andrew,Brooklyn,Bushwick,40.70153,-73.91690,True,...,21.0,1,0.0,NaT,NaN,3.0,1,113,#NAME?,True
7,45073219,Luxurious Quiet Apartment in the Heart of Harlem,77963764786,False,T,Manhattan,Harlem,40.82443,-73.94154,False,...,228.0,2,43.0,2019-06-01,1.65,4.0,1,120,This is a quiet building with absolutely no sm...,True
8,51315866,Glamorous Queen room in the centre of EVERYTH...,92821080349,False,Bianca,Manhattan,Chinatown,40.71427,-73.99261,False,...,92.0,1,49.0,2019-06-30,1.86,5.0,3,111,Please do not smoke and be respectful of other...,True
9,21950075,Sunny garden studio minutes from Prospect Park,84656858323,False,Grace,Brooklyn,Park Slope,40.67331,-73.97471,True,...,90.0,3,2.0,2019-06-23,0.46,4.0,1,5,<NA>,False




--- Resumo das Informações (Info) ---
<class 'pandas.core.frame.DataFrame'>
Index: 100334 entries, 0 to 102598
Data columns (total 24 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   id                              100334 non-null  int64         
 1   name                            100334 non-null  string        
 2   host_id                         100334 non-null  int64         
 3   host_identity_verified          100334 non-null  bool          
 4   host_name                       100334 non-null  string        
 5   neighbourhood_group             100334 non-null  string        
 6   neighbourhood                   100334 non-null  string        
 7   lat                             100334 non-null  float64       
 8   long                            100334 non-null  float64       
 9   instant_bookable                100334 non-null  bool          
 10  cancellation_policy  

## Salvando Dataset

In [ ]:

df.to_csv(data_layer_filepath + 'silver/airbnb-dataset-silver.csv', index=False)

print("Dataset da camada Silver salvo com sucesso!")

Dataset da camada Silver salvo com sucesso!


## Enviando os dados para a base de dados

In [ ]:
# =====================================================================
# ETAPA FINAL: CARGA E VERIFICAÇÃO NO POSTGRESQL (usando Psycopg)
# =====================================================================
import psycopg
import io

print("--- Iniciando processo de carga e verificação no PostgreSQL ---")

# --- CONFIGURAções DO BANCO DE DADOS ---
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5433"
DB_NAME = "airbnb"
DB_SCHEMA = 'silver'
TABLE_NAME = 'listings'
TABLE_FULL_NAME = f'{DB_SCHEMA}.{TABLE_NAME}'

# --- DEFINIÇÃO DA ESTRUTURA DA TABELA (DDL) ---
CREATE_TABLE_SQL = f"""
CREATE SCHEMA IF NOT EXISTS {DB_SCHEMA};
DROP TABLE IF EXISTS {TABLE_FULL_NAME};
CREATE TABLE {TABLE_FULL_NAME} (
    id BIGINT PRIMARY KEY, name TEXT, host_id BIGINT, host_identity_verified BOOLEAN,
    host_name VARCHAR(255), neighbourhood_group VARCHAR(255), neighbourhood VARCHAR(255),
    lat NUMERIC(10, 7), long NUMERIC(10, 7), instant_bookable BOOLEAN,
    cancellation_policy VARCHAR(100), room_type VARCHAR(100), construction_year INTEGER,
    price NUMERIC(10, 2), service_fee NUMERIC(10, 2), minimum_nights INTEGER,
    number_of_reviews INTEGER, last_review DATE, reviews_per_month NUMERIC(5, 2),
    review_rate_number INTEGER, calculated_host_listings_count INTEGER,
    availability_365 INTEGER, house_rules TEXT, has_house_rules BOOLEAN
);
"""

# ===============================================================================
# PONTO DE VERIFICAÇÃO DO DATAFRAME
# ===============================================================================
df_para_carregar = df_sem_duplicados.copy()
print(f"DataFrame '{'df_sem_duplicados'}' selecionado para a carga.")
print(f"Total de linhas a serem carregadas: {len(df_para_carregar)}")
# ===============================================================================

# ===============================================================================
# CORREÇÃO: AJUSTE DE TIPOS DE DADOS ANTES DA CARGA
# Garantimos que as colunas de inteiros não tenham casas decimais.
# ===============================================================================
colunas_inteiras = [
    'construction_year',
    'minimum_nights',
    'number_of_reviews',
    'review_rate_number',
    'calculated_host_listings_count',
    'availability_365'
]

for col in colunas_inteiras:
    # Usamos o tipo Int64Dtype que suporta valores nulos (NaN)
    df_para_carregar[col] = df_para_carregar[col].astype('Int64')

print("\nTipos de dados das colunas de inteiros ajustados com sucesso.")
# ===============================================================================


# --- CONEXÃO E CARGA ---
try:
    with psycopg.connect(f"host={DB_HOST} dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} port={DB_PORT}") as conn:
        with conn.cursor() as cur:
            print("\nConexão com o PostgreSQL estabelecida.")
            cur.execute(CREATE_TABLE_SQL)
            print("Estrutura do banco de dados (re)criada com sucesso.")
            
            buffer = io.StringIO()
            # O .to_csv agora irá gerar '9' em vez de '9.0'
            df_para_carregar.to_csv(buffer, index=False, header=False, sep='\t')
            buffer.seek(0)
            
            print("Iniciando carga de dados em massa com COPY...")
            with cur.copy(f"COPY {TABLE_FULL_NAME} FROM STDIN WITH (FORMAT CSV, DELIMITER E'\\t')") as copy:
                copy.write(buffer.read())
            
    print(f"Carga de dados finalizada.")

except (Exception, psycopg.DatabaseError) as error:
    print(f"ERRO: A transação falhou. Causa: {error}")

finally:
    print("Processo de carga encerrado.")

# =====================================================================
# VERIFICAÇÃO PÓS-CARGA
# =====================================================================
print("\n--- Iniciando verificação pós-carga ---")
try:
    with psycopg.connect(f"host={DB_HOST} dbname={DB_NAME} user={DB_USER} password={DB_PASSWORD} port={DB_PORT}") as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT COUNT(*) FROM {TABLE_FULL_NAME};")
            count_db = cur.fetchone()[0]
            print(f"Contagem de linhas no DataFrame: {len(df_para_carregar)}")
            print(f"Contagem de linhas no Banco de Dados: {count_db}")

            if count_db == len(df_para_carregar):
                print("✅ SUCESSO! O número de linhas é compatível.")
            else:
                print("⚠️ FALHA! O número de linhas é diferente.")

            print("\nBuscando uma amostra de 5 linhas do banco de dados para inspeção visual:")
            df_from_db = pd.read_sql(f"SELECT * FROM {TABLE_FULL_NAME} LIMIT 5", conn)
            
    display(df_from_db)

except (Exception, psycopg.DatabaseError) as error:
    print(f"ERRO durante a verificação: {error}")

<psycopg.Cursor [COMMAND_OK] [INTRANS] (host=localhost port=5433 user=postgres database=airbnb) at 0x7aef22e6a390>